# TA-05: Simulasi Sistem Bengkel (2 Montir, 1 Resepsionis/Kasir)
**Mata Praktikum:** Pemodelan dan Simulasi, Modul 5

**Nama / NIM:** ........................

## 1. Deskripsi Sistem dan Alur Logika
Mobil datang, mendaftar di resepsionis, lalu dicek apakah ada montir bebas. Jika tidak, mobil menunggu di ruang tunggu. Setelah perbaikan selesai, pemilik membayar di kasir (dilayani resepsionis yang sama), lalu mobil keluar.

![Flowchart](flowchart_bengkel.png)

## 2. Asumsi Model
| Komponen | Asumsi |
|---|---|
| Satuan waktu | menit |
| Jumlah mobil | 50 |
| Waktu antar-kedatangan | Eksponensial, rata-rata 12 menit (mobil 1 datang pada menit 0) |
| Waktu pendaftaran | Uniform(2, 4) |
| Waktu perbaikan | Normal(20, 5), minimal 5 menit |
| Waktu pembayaran | Uniform(3, 5) |
| Montir | 2 (dihitung sebagai penghitung `montir_bebas`) |
| Resepsionis/kasir | 1, dipakai untuk pendaftaran **dan** pembayaran |
| Disiplin antrian | FIFO di setiap antrian |
| Prioritas resepsionis | Pembayaran didahulukan dari pendaftaran, agar mobil yang sudah selesai cepat keluar |
| Random seed | 42 (hasil dapat direproduksi) |

## 3. Desain Event Calendar
Setiap event berbentuk `[waktu, tipe, id_mobil]` dan Event Calendar selalu diurutkan berdasarkan waktu.

| Event | Arti | Aksi utama |
|---|---|---|
| `ARRIVAL` | Mobil tiba | Masuk antrian resepsionis, lalu resepsionis dipanggil jika bebas |
| `REG_END` | Pendaftaran selesai | Resepsionis bebas. Mobil ke montir jika ada yang bebas, jika tidak ke ruang tunggu |
| `REPAIR_END` | Perbaikan selesai | Montir bebas. Mobil ke antrian kasir. Mobil di ruang tunggu (jika ada) langsung diperbaiki |
| `PAY_END` | Pembayaran selesai | Resepsionis bebas. Mobil keluar bengkel |

## 4. Implementasi
### 4.1 Parameter dan import

In [1]:
import numpy as np

# =====================================================
# 1. PARAMETER SIMULASI (semua satuan dalam menit)
# =====================================================
JUMLAH_MOBIL = 50          # jumlah mobil yang disimulasikan
JUMLAH_MONTIR = 2          # kapasitas montir
MEAN_IAT = 12              # rata-rata waktu antar-kedatangan (eksponensial)
DAFTAR_MIN, DAFTAR_MAX = 2, 4      # waktu pendaftaran ~ Uniform(2, 4)
REPAIR_MEAN, REPAIR_STD = 20, 5    # waktu perbaikan ~ Normal(20, 5), minimal 5
BAYAR_MIN, BAYAR_MAX = 3, 5        # waktu pembayaran ~ Uniform(3, 5)
SEED = 42

### 4.2 Fungsi bantu
Fungsi untuk menjadwalkan event dan memulai tiap layanan, serta aturan prioritas resepsionis (`layani_resepsionis`).

In [2]:
# =====================================================
# 2. FUNGSI BANTU
# =====================================================
def jadwalkan(s, waktu, tipe, mobil):
    """Tambahkan event ke Event Calendar lalu urutkan berdasarkan waktu."""
    s["kalender"].append([round(waktu, 2), tipe, mobil])
    s["kalender"].sort()


def mulai_pendaftaran(s, mobil):
    d = s["data"][mobil]
    s["resepsionis"] = "BUSY"
    d["tunggu_daftar"] = s["clock"] - d["datang"]
    s["sibuk_resepsionis"] += d["t_daftar"]
    jadwalkan(s, s["clock"] + d["t_daftar"], "REG_END", mobil)


def mulai_pembayaran(s, mobil):
    d = s["data"][mobil]
    s["resepsionis"] = "BUSY"
    d["tunggu_bayar"] = s["clock"] - d["selesai_perbaikan"]
    s["sibuk_resepsionis"] += d["t_bayar"]
    jadwalkan(s, s["clock"] + d["t_bayar"], "PAY_END", mobil)


def mulai_perbaikan(s, mobil):
    d = s["data"][mobil]
    s["montir_bebas"] -= 1
    d["tunggu_montir"] = s["clock"] - d["selesai_daftar"]
    s["sibuk_montir"] += d["t_perbaikan"]
    jadwalkan(s, s["clock"] + d["t_perbaikan"], "REPAIR_END", mobil)


def layani_resepsionis(s):
    """Dipanggil saat resepsionis mungkin bebas.
    Prioritas: pembayaran lebih dulu, baru pendaftaran."""
    if s["resepsionis"] == "IDLE":
        if s["antrian_bayar"]:
            mulai_pembayaran(s, s["antrian_bayar"].pop(0))
        elif s["antrian_daftar"]:
            mulai_pendaftaran(s, s["antrian_daftar"].pop(0))

### 4.3 Event handler
Satu fungsi untuk setiap jenis kejadian, seperti pada contoh `handle_arrival` / `handle_departure` di modul.

In [3]:
# =====================================================
# 3. EVENT HANDLER
# =====================================================
def handle_arrival(s, mobil):
    langsung = (s["resepsionis"] == "IDLE")
    s["antrian_daftar"].append(mobil)
    layani_resepsionis(s)
    return "Langsung didaftar" if langsung else "Antri di resepsionis"


def handle_reg_end(s, mobil):
    s["resepsionis"] = "IDLE"
    s["data"][mobil]["selesai_daftar"] = s["clock"]
    if s["montir_bebas"] > 0:
        mulai_perbaikan(s, mobil)
        ket = "Langsung diperbaiki"
    else:
        s["antrian_tunggu"].append(mobil)
        ket = "Masuk ruang tunggu"
    layani_resepsionis(s)
    return ket


def handle_repair_end(s, mobil):
    s["montir_bebas"] += 1
    s["data"][mobil]["selesai_perbaikan"] = s["clock"]
    s["antrian_bayar"].append(mobil)
    ket = "Menuju kasir"
    if s["antrian_tunggu"]:
        berikut = s["antrian_tunggu"].pop(0)
        mulai_perbaikan(s, berikut)
        ket += f", montir ambil mobil {berikut}"
    layani_resepsionis(s)
    return ket


def handle_payment_end(s, mobil):
    s["resepsionis"] = "IDLE"
    s["data"][mobil]["pergi"] = s["clock"]
    s["selesai"] += 1
    layani_resepsionis(s)
    return "Mobil keluar bengkel"


HANDLER = {
    "ARRIVAL": handle_arrival,
    "REG_END": handle_reg_end,
    "REPAIR_END": handle_repair_end,
    "PAY_END": handle_payment_end,
}

### 4.4 Fungsi utama simulasi (main simulation loop)
Berisi pembangkitan data acak, inisialisasi, loop `while kalender`, validasi internal, dan statistik akhir. Waktu tunggu dirata-ratakan atas **semua** mobil, termasuk yang tidak menunggu (nilai 0).

In [4]:
# =====================================================
# 4. FUNGSI UTAMA SIMULASI
# =====================================================
def simulasi_bengkel(n=JUMLAH_MOBIL, seed=SEED, verbose=True):
    np.random.seed(seed)

    # --- Bangkitkan data acak untuk tiap mobil ---
    iat = np.random.exponential(MEAN_IAT, n - 1)
    t_daftar = np.random.uniform(DAFTAR_MIN, DAFTAR_MAX, n)
    t_perbaikan = np.maximum(5, np.random.normal(REPAIR_MEAN, REPAIR_STD, n))
    t_bayar = np.random.uniform(BAYAR_MIN, BAYAR_MAX, n)
    waktu_datang = np.concatenate(([0.0], np.cumsum(iat)))

    # --- Inisialisasi state ---
    s = {
        "clock": 0.0,
        "resepsionis": "IDLE",
        "montir_bebas": JUMLAH_MONTIR,
        "antrian_daftar": [],
        "antrian_tunggu": [],
        "antrian_bayar": [],
        "kalender": [],
        "data": {},
        "selesai": 0,
        "sibuk_montir": 0.0,
        "sibuk_resepsionis": 0.0,
    }
    for i in range(n):
        mobil = i + 1
        s["data"][mobil] = {
            "datang": round(waktu_datang[i], 2),
            "t_daftar": round(t_daftar[i], 2),
            "t_perbaikan": round(t_perbaikan[i], 2),
            "t_bayar": round(t_bayar[i], 2),
        }
        s["kalender"].append([s["data"][mobil]["datang"], "ARRIVAL", mobil])
    s["kalender"].sort()

    # --- Variabel statistik panjang antrian ---
    area_tunggu = 0.0      # luas (panjang antrian x waktu), ruang tunggu montir
    area_resepsionis = 0.0  # antrian di resepsionis (daftar + bayar)
    maks_tunggu = 0
    maks_resepsionis = 0
    waktu_terakhir = 0.0

    if verbose:
        print("=== SIMULASI BENGKEL (2 Montir, 1 Resepsionis/Kasir) ===\n")
        print(f"{'Clock':<9}{'Event':<12}{'Mobil':<7}{'Q.Daftar':<10}"
              f"{'R.Tunggu':<10}{'Q.Bayar':<9}{'Montir':<8}{'Resp.':<7}Keterangan")
        print("-" * 100)

    # --- Main simulation loop ---
    while s["kalender"]:
        waktu, tipe, mobil = s["kalender"].pop(0)

        # akumulasi luas panjang antrian sebelum clock maju
        selang = waktu - waktu_terakhir
        area_tunggu += len(s["antrian_tunggu"]) * selang
        area_resepsionis += (len(s["antrian_daftar"]) + len(s["antrian_bayar"])) * selang
        waktu_terakhir = waktu
        s["clock"] = waktu

        ket = HANDLER[tipe](s, mobil)

        maks_tunggu = max(maks_tunggu, len(s["antrian_tunggu"]))
        maks_resepsionis = max(maks_resepsionis,
                               len(s["antrian_daftar"]) + len(s["antrian_bayar"]))

        if verbose:
            print(f"{waktu:<9.2f}{tipe:<12}{mobil:<7}{len(s['antrian_daftar']):<10}"
                  f"{len(s['antrian_tunggu']):<10}{len(s['antrian_bayar']):<9}"
                  f"{s['montir_bebas']:<8}{s['resepsionis']:<7}{ket}")

    # =================================================
    # 5. VALIDASI INTERNAL
    # =================================================
    assert s["selesai"] == n, "Tidak semua mobil selesai"
    assert not (s["antrian_daftar"] or s["antrian_tunggu"] or s["antrian_bayar"])
    assert s["montir_bebas"] == JUMLAH_MONTIR and s["resepsionis"] == "IDLE"
    for d in s["data"].values():
        total = (d["tunggu_daftar"] + d["t_daftar"] + d["tunggu_montir"]
                 + d["t_perbaikan"] + d["tunggu_bayar"] + d["t_bayar"])
        assert abs((d["pergi"] - d["datang"]) - total) < 0.1, "Waktu tidak konsisten"

    # =================================================
    # 6. STATISTIK AKHIR
    # =================================================
    T = s["clock"]
    data = list(s["data"].values())
    rata = lambda kunci: sum(d[kunci] for d in data) / n
    hasil = {
        "mobil_selesai": s["selesai"],
        "waktu_simulasi": T,
        "rata_tunggu_daftar": rata("tunggu_daftar"),
        "rata_tunggu_montir": rata("tunggu_montir"),
        "maks_tunggu_montir": max(d["tunggu_montir"] for d in data),
        "rata_tunggu_bayar": rata("tunggu_bayar"),
        "rata_waktu_di_bengkel": sum(d["pergi"] - d["datang"] for d in data) / n,
        "persen_menunggu_montir": 100 * sum(d["tunggu_montir"] > 0 for d in data) / n,
        "util_montir": 100 * s["sibuk_montir"] / (JUMLAH_MONTIR * T),
        "util_resepsionis": 100 * s["sibuk_resepsionis"] / T,
        "Lq_ruang_tunggu": area_tunggu / T,
        "maks_ruang_tunggu": maks_tunggu,
        "maks_antrian_resepsionis": maks_resepsionis,
    }

    if verbose:
        print("\n=== HASIL SIMULASI ===")
        print(f"Total mobil selesai                  : {hasil['mobil_selesai']}")
        print(f"Total waktu simulasi                 : {T:.2f} menit")
        print(f"Rata-rata tunggu pendaftaran         : {hasil['rata_tunggu_daftar']:.2f} menit")
        print(f"Rata-rata tunggu montir (ruang tunggu): {hasil['rata_tunggu_montir']:.2f} menit")
        print(f"Tunggu montir maksimum               : {hasil['maks_tunggu_montir']:.2f} menit")
        print(f"Rata-rata tunggu kasir               : {hasil['rata_tunggu_bayar']:.2f} menit")
        print(f"Rata-rata waktu total di bengkel     : {hasil['rata_waktu_di_bengkel']:.2f} menit")
        print(f"Mobil yang harus menunggu montir     : {hasil['persen_menunggu_montir']:.1f} %")
        print(f"Utilitas montir                      : {hasil['util_montir']:.1f} %")
        print(f"Utilitas resepsionis/kasir           : {hasil['util_resepsionis']:.1f} %")
        print(f"Rata-rata panjang antrian ruang tunggu: {hasil['Lq_ruang_tunggu']:.2f} mobil")
        print(f"Antrian ruang tunggu maksimum        : {hasil['maks_ruang_tunggu']} mobil")
        print(f"Antrian resepsionis maksimum         : {hasil['maks_antrian_resepsionis']} mobil")
        print("\nValidasi internal: semua mobil selesai, antrian kosong, dan "
              "waktu tiap mobil konsisten (OK).")
    return hasil

## 5. Menjalankan Simulasi (seed = 42)

In [5]:
simulasi_bengkel()

=== SIMULASI BENGKEL (2 Montir, 1 Resepsionis/Kasir) ===

Clock    Event       Mobil  Q.Daftar  R.Tunggu  Q.Bayar  Montir  Resp.  Keterangan
----------------------------------------------------------------------------------------------------
0.00     ARRIVAL     1      0         0         0        2       BUSY   Langsung didaftar
2.37     REG_END     1      0         0         0        1       IDLE   Langsung diperbaiki
5.63     ARRIVAL     2      0         0         0        1       BUSY   Langsung didaftar
9.57     REG_END     2      0         0         0        0       IDLE   Langsung diperbaiki
15.28    REPAIR_END  1      0         0         0        1       BUSY   Menuju kasir
19.74    PAY_END     1      0         0         0        1       IDLE   Mobil keluar bengkel
34.78    REPAIR_END  2      0         0         0        2       BUSY   Menuju kasir
38.52    PAY_END     2      0         0         0        2       IDLE   Mobil keluar bengkel
41.75    ARRIVAL     3      0         

## 6. Replikasi (Opsional): Seberapa Stabil Hasilnya?
Satu kali run hanya satu sampel acak. Berikut 10 replikasi dengan seed berbeda.

In [6]:
import pandas as pd

hasil_rep = [simulasi_bengkel(seed=k, verbose=False) for k in range(1, 11)]
df = pd.DataFrame(hasil_rep)[["rata_tunggu_montir", "rata_waktu_di_bengkel",
                              "util_montir", "util_resepsionis", "maks_ruang_tunggu"]]
df.index = [f"seed {k}" for k in range(1, 11)]
print(df.round(2).to_string())
print("\nRata-rata 10 replikasi:")
print(df.mean().round(2).to_string())

         rata_tunggu_montir  rata_waktu_di_bengkel  util_montir  util_resepsionis  maks_ruang_tunggu
seed 1                28.62                  61.01        89.31             58.53                  7
seed 2                62.18                  94.19        96.59             66.12                 13
seed 3                30.47                  60.35        89.81             62.97                  9
seed 4                11.05                  39.95        71.72             50.84                  4
seed 5                21.92                  52.60        82.25             55.86                  8
seed 6                 4.83                  32.94        66.14             46.81                  3
seed 7                 9.47                  39.32        83.41             59.50                  4
seed 8                 8.82                  36.97        72.57             51.55                  4
seed 9                11.10                  42.52        76.59             48.55          